# BIOT 6900 — PDAC data preparation

Downloads the CPTAC-PDAC files from LinkedOmics and converts them to the gene-level format needed for the project.

**Comparison used:** Tumor vs Normal for RNA and protein.  
**Genomic evidence:** gene-level somatic mutation frequency.

**Disclosure:** LLM tools have been used to gather the data.

In [2]:
from pathlib import Path
import requests
import numpy as np
import pandas as pd
from scipy import stats

data = Path("../data/pdac")
raw = data / "raw"
raw.mkdir(parents=True, exist_ok=True)

base = "https://linkedomics.org/data_download/CPTAC-PDAC/"
files = [
    "mRNA_RSEM_UQ_log2_Tumor.cct",
    "mRNA_RSEM_UQ_log2_Normal.cct",
    "proteomics_gene_level_MD_abundance_tumor.cct",
    "proteomics_gene_level_MD_abundance_normal.cct",
    "Mutation_gene_level.cgt",
]

for name in files:
    path = raw / name
    if path.exists():
        print("Already have:", name)
        continue

    print("Downloading:", name)
    with requests.get(base + name, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)

print("Done.")

Downloading: mRNA_RSEM_UQ_log2_Tumor.cct
Downloading: mRNA_RSEM_UQ_log2_Normal.cct
Downloading: proteomics_gene_level_MD_abundance_tumor.cct
Downloading: proteomics_gene_level_MD_abundance_normal.cct
Downloading: Mutation_gene_level.cgt
Done.


In [3]:
# Load raw matrices: genes x samples
rna_t = pd.read_csv(raw / "mRNA_RSEM_UQ_log2_Tumor.cct", sep="\t", index_col=0)
rna_n = pd.read_csv(raw / "mRNA_RSEM_UQ_log2_Normal.cct", sep="\t", index_col=0)

prot_t = pd.read_csv(raw / "proteomics_gene_level_MD_abundance_tumor.cct", sep="\t", index_col=0)
prot_n = pd.read_csv(raw / "proteomics_gene_level_MD_abundance_normal.cct", sep="\t", index_col=0)

mut = pd.read_csv(raw / "Mutation_gene_level.cgt", sep="\t", index_col=0)

print("RNA tumor/normal:", rna_t.shape, rna_n.shape)
print("Protein tumor/normal:", prot_t.shape, prot_n.shape)
print("Mutation:", mut.shape)

RNA tumor/normal: (28057, 140) (28057, 21)
Protein tumor/normal: (11662, 140) (11662, 75)
Mutation: (4424, 140)


In [5]:
# One row per gene: effect = mean(tumor) - mean(normal), plus Welch t-test p-value ==> since classes are of different size & variance
def differential(tumor, normal):
    common = tumor.index.intersection(normal.index)
    tumor = tumor.loc[common].apply(pd.to_numeric, errors="coerce")
    normal = normal.loc[common].apply(pd.to_numeric, errors="coerce")
    _, p = stats.ttest_ind(
        tumor.to_numpy(),
        normal.to_numpy(),
        axis=1,
        equal_var=False,
        nan_policy="omit"
    )
    return pd.DataFrame({
        "gene": common,
        "effect": tumor.mean(axis=1).to_numpy() - normal.mean(axis=1).to_numpy(),
        "pval": p
    })

tx = differential(rna_t, rna_n)
pr = differential(prot_t, prot_n)

# Gene-level somatic mutation frequency = genomic association strength for this exercise
observed = mut.notna()
mutated = observed & (mut.map(lambda x: str(x).strip() if pd.notna(x) else x) != "WT")
mut_freq = mutated.sum(axis=1) / observed.sum(axis=1).replace(0, np.nan)

gw = mut_freq.rename("association").reset_index()
gw.columns = ["gene", "association"]

display(tx.head())
display(pr.head())
display(gw.head())

C:\Users\mattighattabhojann.s\AppData\Local\Temp\ipykernel_16032\2908140146.py:6: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  _, p = stats.ttest_ind(


,gene,effect,pval
0,A1BG,-0.479548,1.250690e-02
1,A1BG-AS1,-0.178861,4.499880e-01
2,A1CF,-1.463168,2.552527e-08
3,A2M,-0.624133,5.580447e-07
4,A2M-AS1,-0.480572,8.412509e-03


,gene,effect,pval
0,A1BG,0.303070,2.094465e-04
1,A1CF,-1.868882,6.235209e-29
2,A2M,0.228212,1.502368e-02
3,A2ML1,0.884157,2.971118e-08
4,A4GALT,0.041078,9.217133e-01


,gene,association
0,A1BG,0.007143
1,A2ML1,0.007143
2,A4GNT,0.007143
3,AAGAB,0.007143
4,AARS,0.007143


In [6]:
# Save the 3 files used by the project notebook
tx.to_csv(data / "pdac_transcriptomics.tsv", sep="\t", index=False)
pr.to_csv(data / "pdac_proteomics.tsv", sep="\t", index=False)
gw.to_csv(data / "pdac_genomics.tsv", sep="\t", index=False)

print("Saved:")
print(data / "pdac_transcriptomics.tsv")
print(data / "pdac_proteomics.tsv")
print(data / "pdac_genomics.tsv")

Saved:
..\data\pdac\pdac_transcriptomics.tsv
..\data\pdac\pdac_proteomics.tsv
..\data\pdac\pdac_genomics.tsv


**I used tumor vs normal for both RNA and protein so the biological comparison is consistent across the two expression layers. I did not use the 7-sample Duct RNA set because there is no matching Duct proteomics matrix on the LinkedOmics PDAC download page. For genomics, I used gene-level somatic mutation frequency as the association-strength measure, following the same idea used to collapse CPTAC mutation data in the class exercise.**